In [ ]:
# C?lula 1: Configura??es iniciais e imports
import os
import json
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw
import cv2
import torch
from ultralytics import YOLO
import pandas as pd
import seaborn as sns
from tqdm import tqdm
import shutil
import yaml

# Configura??es globais
BASE_DIR = './'
IMAGE_DIR = os.path.join(BASE_DIR, "antelope")
ANNOTATIONS_FILE = os.path.join(BASE_DIR, "antelope.json")
OUTPUT_DIR = os.path.join(BASE_DIR, "processed")
RESULTS_DIR = os.path.join(BASE_DIR, "results")
TRAIN_DIR = os.path.join(BASE_DIR, "train_data")
YOLO_CONFIG = os.path.join(BASE_DIR, "antelope_pose.yaml")
TRAIN_MODEL_PATH = os.path.join(BASE_DIR, "yolov8n-pose.pt")
TRAINED_MODEL_PATH = os.path.join(BASE_DIR, "yolov8n-pose-antelope.pt")

# Criar diret?rios necess?rios
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(TRAIN_DIR, exist_ok=True)

In [ ]:
def create_yolo_config():
    config = f"""
# Caminhos
path: {BASE_DIR}
train: {TRAIN_DIR}
val: {TRAIN_DIR}  # Na pr?tica usar conjunto diferente

# Keypoints
kpt_shape: [8, 3]  # 8 keypoints com (x, y, visibility)

# Classes
names:
  0: antelope

# Par?metros obrigat?rios
nc: 1  # n?mero de classes
flip_idx: [1, 0, 3, 2, 5, 4, 7, 6]  # conex?es para flip horizontal
    """
    
    with open(YOLO_CONFIG, 'w') as f:
        f.write(config)
        
    print(f"Configura??o YAML criada em: {YOLO_CONFIG}")

create_yolo_config()

In [ ]:
# C?lula 3: Prepara??o do dataset no formato YOLO
def prepare_yolo_dataset():
    """Converte anota??es para o formato YOLOv8 pose"""
    with open(ANNOTATIONS_FILE) as f:
        annotations = json.load(f)

    print(f"Preparando dataset YOLO em {TRAIN_DIR}...")
    
    for img_name, bboxes in tqdm(annotations.items(), desc="Convertendo anota??es"):
        try:
            # Copiar imagem
            src_img = os.path.join(IMAGE_DIR, img_name)
            dst_img = os.path.join(TRAIN_DIR, img_name)
            shutil.copy(src_img, dst_img)
            
            # Criar arquivo de anota??es
            txt_path = os.path.join(TRAIN_DIR, os.path.splitext(img_name)[0] + '.txt')
            
            with open(txt_path, 'w') as f_txt:
                for bbox in bboxes:
                    img = Image.open(src_img)
                    img_w, img_h = img.size
                    
                    # Converter bounding box
                    x_center = (bbox['bndbox']['xmin'] + bbox['bndbox']['xmax']) / (2 * img_w)
                    y_center = (bbox['bndbox']['ymin'] + bbox['bndbox']['ymax']) / (2 * img_h)
                    width = (bbox['bndbox']['xmax'] - bbox['bndbox']['xmin']) / img_w
                    height = (bbox['bndbox']['ymax'] - bbox['bndbox']['ymin']) / img_h
                    
                    # Gerar keypoints fict?cios (substituir por seus dados reais)
                    keypoints = [
                        0.7, 0.2, 2,  # snout
                        0.6, 0.15, 2,  # head
                        0.5, 0.3, 2,   # neck_base
                        0.3, 0.7, 2,   # front_right_leg
                        0.7, 0.7, 2,   # front_left_leg
                        0.2, 0.8, 2,   # hind_right_leg
                        0.8, 0.8, 2,   # hind_left_leg
                        0.1, 0.4, 2    # tail_base
                    ]
                    
                    line = [0, x_center, y_center, width, height] + keypoints
                    f_txt.write(' '.join(map(str, line)) + '\n')
                    
        except Exception as e:
            print(f"Erro ao processar {img_name}: {e}")
    
    print("Convers?o conclu?da!")

prepare_yolo_dataset()

In [ ]:
def train_model():
    """Executa o treinamento com transfer learning"""
    print("\nIniciando treinamento com transfer learning...")
    
    # Verificar dados
    if not os.path.exists(TRAIN_DIR) or len(os.listdir(TRAIN_DIR)) == 0:
        raise FileNotFoundError("Dados de treino n?o encontrados!")
    
    # Carregar modelo
    model = YOLO(TRAIN_MODEL_PATH)
    
    # Hiperpar?metros atualizados para YOLOv8.3+
    train_args = {
        'data': YOLO_CONFIG,
        'epochs': 30,
        'imgsz': 256,
        'batch': 8,
        'optimizer': 'Adam',
        'lr0': 0.001,
        'pretrained': True,
        'project': RESULTS_DIR,
        'name': 'antelope_pose',
        'exist_ok': True,
        'device': '0' if torch.cuda.is_available() else 'cpu',
        'freeze': 5,  # Congelar primeiras 5 camadas
        'weight_decay': 0.0005,
        'patience': 4,  # Early stopping
        'label_smoothing': 0.1,
        'fliplr': 0.5,  # Aumento de dados
        'mosaic': 1.0,   # Aumento de dados
    }
    
    try:
        # Treinar modelo
        results = model.train(**train_args)
        
        # Salvar modelo
        model.save(TRAINED_MODEL_PATH)
        
        # Plotar m?tricas
        results.plot()
        plt.savefig(os.path.join(RESULTS_DIR, 'training_metrics.png'))
        plt.close()
        
        print(f"? Treinamento conclu?do! Modelo salvo em: {TRAINED_MODEL_PATH}")
        
        # Mostrar estat?sticas
        total_params = sum(p.numel() for p in model.parameters())
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f"\nPar?metros trein?veis: {trainable_params/total_params:.2%}")
        
        return model
        
    except Exception as e:
        print(f"? Erro durante o treinamento: {e}")
        print("Solu??es potenciais:")
        print("1. Verifique o formato das anota??es nos arquivos .txt")
        print("2. Atualize o Ultralytics: pip install --upgrade ultralytics")
        print("3. Verifique o caminho absoluto das imagens no YAML")
        raise

# Executar treinamento
train_model()

In [ ]:
# C?lula 5: Valida??o do Modelo
def validate_model(model):
    """Avalia o modelo no conjunto de valida??o"""
    print("\nIniciando valida??o do modelo...")
    
    metrics = model.val(
        data=YOLO_CONFIG,
        batch=4,
        imgsz=256,
        conf=0.5,
        iou=0.6,
        device='0' if torch.cuda.is_available() else 'cpu'
    )
    
    # Salvar m?tricas
    with open(os.path.join(RESULTS_DIR, 'validation_metrics.json'), 'w') as f:
        json.dump(metrics.results_dict, f)
    
    print(f"? Valida??o conclu?da! Precis?o mAP50-95: {metrics.box.map:.2f}")
    return metrics

# Executar valida??o se o modelo existe
if os.path.exists(TRAINED_MODEL_PATH):
    model = YOLO(TRAINED_MODEL_PATH)
    val_metrics = validate_model(model)
else:
    print("Modelo treinado n?o encontrado. Pulando valida??o.")

In [ ]:
def train_model():
    """Executa o treinamento com transfer learning"""
    print("\nIniciando treinamento com transfer learning...")
    
    # Verificar dados
    if not os.path.exists(TRAIN_DIR) or len(os.listdir(TRAIN_DIR)) == 0:
        raise FileNotFoundError("Dados de treino n?o encontrados!")
    
    # Carregar modelo
    model = YOLO(TRAIN_MODEL_PATH)
    
    # Congelar backbone
    for name, param in model.named_parameters():
        if 'backbone' in name:
            param.requires_grad = False
    
    # Hiperpar?metros com early stopping
    train_args = {
        'data': YOLO_CONFIG,
        'epochs': 30,
        'imgsz': 256,
        'batch': 8,
        'optimizer': 'Adam',
        'lr0': 0.001,
        'pretrained': True,
        'project': RESULTS_DIR,
        'name': 'antelope_pose_frozen_backbone',
        'exist_ok': True,
        'device': '0' if torch.cuda.is_available() else 'cpu',
        'freeze': [0, 1, 2, 3, 4],
        'weight_decay': 0.0005,
        'momentum': 0.9,
        'label_smoothing': 0.1,
        'patience': 4,
        'save_period': 1
    }
    
    # Callback corrigido
    class CustomCallback:
        def __init__(self):
            self.best_map = 0.0
            
        def on_train_epoch_end(self, trainer):
            current_map = trainer.validator.metrics.ap50
            print(f"\n?poca {trainer.epoch + 1}/{trainer.args.epochs}")
            print(f"mAP50: {current_map:.3f} | Best mAP50: {self.best_map:.3f}")
            
            if current_map > self.best_map:
                self.best_map = current_map
                print("?? Novo melhor modelo encontrado!")
    
    # Treinar
    try:
        results = model.train(
            **train_args,
            callbacks={'on_train_epoch_end': CustomCallback().on_train_epoch_end}
        )
    except Exception as e:
        print(f"Erro durante o treinamento: {e}")
        raise
    
    # Salvar modelo
    model.save(TRAINED_MODEL_PATH)
    
    # Plotar m?tricas corrigido
    if results:
        fig = plt.figure(figsize=(15, 8))
        plt.plot(results.metrics['train/box_loss'], label='Train Loss')
        plt.plot(results.metrics['val/box_loss'], label='Val Loss')
        plt.title('Curva de Aprendizado')
        plt.xlabel('?poca')
        plt.ylabel('Perda')
        plt.legend()
        plt.savefig(os.path.join(RESULTS_DIR, 'training_metrics_frozen.png'))
        plt.close()
    
    print(f"Treinamento conclu?do! Modelo salvo em: {TRAINED_MODEL_PATH}")
    
    # Mostrar par?metros trein?veis
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\nPar?metros totais: {total_params:,}")
    print(f"Par?metros trein?veis: {trainable_params:,} ({trainable_params/total_params:.2%})")
    
    return model

# Executar treinamento
train_model()

In [ ]:
# C?lula 7: An?lise Comparativa
def compare_models():
    """Compara o modelo treinado com o modelo base"""
    print("\nIniciando an?lise comparativa...")
    
    # Carregar ambos os modelos
    base_model = YOLO(TRAIN_MODEL_PATH)
    trained_model = YOLO(TRAINED_MODEL_PATH)
    
    # M?tricas de compara??o
    comparison = {
        'base_model': {
            'params': sum(p.numel() for p in base_model.parameters()),
            'size_mb': os.path.getsize(TRAIN_MODEL_PATH) / (1024 * 1024)
        },
        'trained_model': {
            'params': sum(p.numel() for p in trained_model.parameters()),
            'size_mb': os.path.getsize(TRAINED_MODEL_PATH) / (1024 * 1024)
        }
    }
    
    # Plotar compara??o
    plt.figure(figsize=(10, 5))
    
    # Tamanho do modelo
    plt.subplot(1, 2, 1)
    plt.bar(['Base', 'Treinado'], 
            [comparison['base_model']['size_mb'], comparison['trained_model']['size_mb']])
    plt.title('Compara??o de Tamanho dos Modelos')
    plt.ylabel('Tamanho (MB)')
    
    # N?mero de par?metros
    plt.subplot(1, 2, 2)
    plt.bar(['Base', 'Treinado'], 
            [comparison['base_model']['params'], comparison['trained_model']['params']])
    plt.title('N?mero de Par?metros')
    plt.ylabel('Par?metros (milh?es)')
    plt.ticklabel_format(style='sci', axis='y', scilimits=(0,0))
    
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, 'model_comparison.png'))
    plt.close()
    
    print("? An?lise comparativa conclu?da!")
    return comparison

model_comparison = compare_models()

In [ ]:
# C?lula 8: Deploy do Modelo
def prepare_for_deployment(model):
    """Prepara o modelo para produ??o"""
    print("\nPreparando para deployment...")
    
    # Exportar para formatos diferentes
    model.export(format='onnx', imgsz=256, simplify=True)
    model.export(format='engine', imgsz=256)
    
    # Criar arquivo de configura??o
    deploy_config = {
        'model_path': TRAINED_MODEL_PATH,
        'input_size': 256,
        'class_names': ['antelope'],
        'keypoints': [
            'snout', 'head', 'neck_base',
            'front_right_leg', 'front_left_leg',
            'hind_right_leg', 'hind_left_leg',
            'tail_base'
        ]
    }
    
    with open(os.path.join(BASE_DIR, 'deploy_config.json'), 'w') as f:
        json.dump(deploy_config, f, indent=4)
    
    print("? Modelo pronto para deployment!")
    print(f"Formatos dispon?veis: {os.listdir(os.path.dirname(TRAINED_MODEL_PATH))}")
    
prepare_for_deployment(model)